In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
num = 2
device = qml.device('default.qubit',wires=num)

In [3]:
weightage = [0.3,1.1,0.8]
observable = [qml.PauliX(0),qml.PauliZ(1),qml.PauliZ(0)@qml.PauliZ(1)]
hamil = qml.Hamiltonian(weightage,observable)

In [4]:
def ansatz(theta):
    qml.RX(theta[0],wires=0)
    qml.RY(theta[1],wires=1)
    qml.CNOT(wires=[0,1])
    qml.RY(theta[2],wires=0)
    qml.RX(theta[3],wires=1)

In [6]:
@qml.qnode(device)
def energy(theta):
    ansatz(theta)
    return qml.expval(hamil)

In [10]:
np.random.seed(42)
theta_g = np.random.randn(4,requires_grad=True)
optimiser = qml.GradientDescentOptimizer(stepsize=0.3)
for i in range(150):
    theta_g,energies = optimiser.step_and_cost(energy,theta_g)
print(f'energy of ground state is {energies} and theta_g is {theta_g}')

energy of ground state is -1.9544003745317529 and theta_g is [ 2.85398814e-17 -1.44266592e-53 -3.58770670e-01  3.14159265e+00]


In [12]:
@qml.qnode(device)
def overlap_node(theta,theta_g):
    ansatz(theta)
    qml.adjoint(ansatz)(theta_g)
    return qml.probs(wires=[0,1])

In [13]:
def energy_excited(theta,theta_g,a=4.0):
    energies = energy(theta)
    over_square = overlap_node(theta,theta_g)[0]
    return energies+over_square*a

In [14]:
theta_e = np.random.randn(4, requires_grad=True)
cost = lambda p : energy_excited(theta_e,theta_g,a=4.0)

In [16]:
for i in range(150):
    theta_e,energies = optimiser.step_and_cost(cost,theta_e)
energy_excited_state = energy(theta_e)
overlap = overlap_node(theta_e,theta_g)[0]
print(f'energy of first excited state is {energy_excited_state} and overlap is {overlap}')

energy of first excited state is 0.9107875622022539 and overlap is 0.09185372195219481
